In [ ]:
import torch
import random
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
! pip install -q kaggle

In [ ]:
from google.colab import files

files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"beojkk","key":"f5032ed5341afd8bd1744192d3316d76"}'}

In [ ]:
!mkdir ./kaggle/
! cp kaggle.json ~/.kaggle/
! chmod 600 ~/.kaggle/kaggle.json

mkdir: cannot create directory ‘./kaggle/’: File exists


https://www.kaggle.com/datasets/gregorut/videogamesales

In [ ]:
!kaggle datasets download -d gregorut/videogamesales

  0% 0.00/381k [00:00<?, ?B/s]
100% 381k/381k [00:00<00:00, 81.9MB/s]


In [ ]:
! mkdir dataset

In [ ]:
! unzip -qq videogamesales.zip -d dataset 

In [ ]:
import csv

with open('/content/dataset/vgsales.csv', newline='') as f:
    reader = csv.reader(f)
    data = list(reader)[1:]

examples = [data[i][1].lower() for i in range(len(data)) if i % 8 == 0 and len(data[i][0]) < 15]

chars = sorted(set(' '.join(examples)))

In [ ]:
data_size, vocab_size, max_len = len(examples), len(chars), len(max(examples, key=len))
ch_to_idx = {c:i for i, c in enumerate(chars)}
idx_to_ch = {i:c for i, c in enumerate(chars)}

In [ ]:
len(examples)

2075

In [ ]:
max_len

109

In [ ]:
ch_to_idx

{' ': 0,
 '!': 1,
 '$': 2,
 '&': 3,
 "'": 4,
 '(': 5,
 ')': 6,
 '*': 7,
 '+': 8,
 ',': 9,
 '-': 10,
 '.': 11,
 '/': 12,
 '0': 13,
 '1': 14,
 '2': 15,
 '3': 16,
 '4': 17,
 '5': 18,
 '6': 19,
 '7': 20,
 '8': 21,
 '9': 22,
 ':': 23,
 ';': 24,
 '?': 25,
 '@': 26,
 'a': 27,
 'b': 28,
 'c': 29,
 'd': 30,
 'e': 31,
 'f': 32,
 'g': 33,
 'h': 34,
 'i': 35,
 'j': 36,
 'k': 37,
 'l': 38,
 'm': 39,
 'n': 40,
 'o': 41,
 'p': 42,
 'q': 43,
 'r': 44,
 's': 45,
 't': 46,
 'u': 47,
 'v': 48,
 'w': 49,
 'x': 50,
 'y': 51,
 'z': 52,
 '~': 53,
 'à': 54,
 'ä': 55,
 'é': 56,
 'ë': 57}

In [ ]:
X = torch.zeros(size=(data_size, max_len, vocab_size))
Y = torch.zeros(size=(data_size, max_len), dtype=torch.long)
print_once = False

# Make items same size
for i in range(len(examples)):
    while len(examples[i]) < max_len:
        examples[i] += ' '

for example_idx, example in enumerate(examples):
    text_x = example[:-1]
    for i, char in enumerate(text_x):
        X[example_idx][i][ch_to_idx[char]] = 1

    text_y = example[1:]
    for i, char in enumerate(text_y):
        Y[example_idx][i] = ch_to_idx[char]

In [ ]:
class RNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(input_size=vocab_size, hidden_size=100, batch_first=True)
        self.fc = nn.Linear(100, vocab_size)
    
    def forward(self, x):
        batch_size = x.size(0)
        hidden = self.init_hidden(batch_size)
        out, hidden = self.rnn(x, hidden)
        out = out.contiguous().view(-1, 100)
        out = self.fc(out)
        
        return out, hidden
    
    def init_hidden(self, batch_size):
        hidden = torch.zeros(1, batch_size, 100)
        return hidden

In [ ]:
def train(model, lossfunction, optimizer, traindata, epochs):

    for i in range(epochs):
        model.train()
        optimizer.zero_grad()
        output, hidden = model(traindata)
        loss = lossfunction(output, Y.view(-1).long())
        loss.backward()
        optimizer.step()
        
        if (i + 1) % 100 == 0:
            print(f'Epoch:{i+1}/{epochs}, Train Loss: {loss:.4f}')

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
model = RNN()
model.to(device)

RNN(
  (rnn): RNN(58, 100, batch_first=True)
  (fc): Linear(in_features=100, out_features=58, bias=True)
)

In [ ]:
train(model, nn.CrossEntropyLoss(), optim.Adam(model.parameters(), lr=0.01), X, 2000)

Epoch:100/2000, Train Loss: 0.6888
Epoch:200/2000, Train Loss: 0.6531
Epoch:300/2000, Train Loss: 0.6016
Epoch:400/2000, Train Loss: 0.5774
Epoch:500/2000, Train Loss: 0.5460
Epoch:600/2000, Train Loss: 0.5311
Epoch:700/2000, Train Loss: 0.5204
Epoch:800/2000, Train Loss: 0.5101
Epoch:900/2000, Train Loss: 0.5026
Epoch:1000/2000, Train Loss: 0.4959
Epoch:1100/2000, Train Loss: 0.4874
Epoch:1200/2000, Train Loss: 0.4794
Epoch:1300/2000, Train Loss: 0.4729
Epoch:1400/2000, Train Loss: 0.4672
Epoch:1500/2000, Train Loss: 0.4610
Epoch:1600/2000, Train Loss: 0.4560
Epoch:1700/2000, Train Loss: 0.4476
Epoch:1800/2000, Train Loss: 0.4490
Epoch:1900/2000, Train Loss: 0.4384
Epoch:2000/2000, Train Loss: 0.4343


In [ ]:
model_2 = RNN()
model_2.to(device)

train(model_2, nn.CrossEntropyLoss(), optim.Adam(model_2.parameters(), lr=0.001), X, 5000)

Epoch:100/5000, Train Loss: 1.0359
Epoch:200/5000, Train Loss: 0.7285
Epoch:300/5000, Train Loss: 0.6945
Epoch:400/5000, Train Loss: 0.6632
Epoch:500/5000, Train Loss: 0.6383
Epoch:600/5000, Train Loss: 0.6196
Epoch:700/5000, Train Loss: 0.6046
Epoch:800/5000, Train Loss: 0.5925
Epoch:900/5000, Train Loss: 0.5821
Epoch:1000/5000, Train Loss: 0.5727
Epoch:1100/5000, Train Loss: 0.5641
Epoch:1200/5000, Train Loss: 0.5561
Epoch:1300/5000, Train Loss: 0.5484
Epoch:1400/5000, Train Loss: 0.5409
Epoch:1500/5000, Train Loss: 0.5335
Epoch:1600/5000, Train Loss: 0.5259
Epoch:1700/5000, Train Loss: 0.5183
Epoch:1800/5000, Train Loss: 0.5107
Epoch:1900/5000, Train Loss: 0.5030
Epoch:2000/5000, Train Loss: 0.4955
Epoch:2100/5000, Train Loss: 0.4881
Epoch:2200/5000, Train Loss: 0.4805
Epoch:2300/5000, Train Loss: 0.4732
Epoch:2400/5000, Train Loss: 0.4658
Epoch:2500/5000, Train Loss: 0.4586
Epoch:2600/5000, Train Loss: 0.4517
Epoch:2700/5000, Train Loss: 0.4448
Epoch:2800/5000, Train Loss: 0.4381
E

In [ ]:
model_3 = RNN()
model_3.to(device)

train(model_3, nn.CrossEntropyLoss(), optim.Adadelta(model_3.parameters(), lr=0.01), X, 2000)

Epoch:100/2000, Train Loss: 3.5452
Epoch:200/2000, Train Loss: 1.1704
Epoch:300/2000, Train Loss: 1.0966
Epoch:400/2000, Train Loss: 1.0491
Epoch:500/2000, Train Loss: 0.9887
Epoch:600/2000, Train Loss: 0.9184
Epoch:700/2000, Train Loss: 0.8797
Epoch:800/2000, Train Loss: 0.8635
Epoch:900/2000, Train Loss: 0.8497
Epoch:1000/2000, Train Loss: 0.8374
Epoch:1100/2000, Train Loss: 0.8257
Epoch:1200/2000, Train Loss: 0.8144
Epoch:1300/2000, Train Loss: 0.8034
Epoch:1400/2000, Train Loss: 0.7932
Epoch:1500/2000, Train Loss: 0.7842
Epoch:1600/2000, Train Loss: 0.7766
Epoch:1700/2000, Train Loss: 0.7704
Epoch:1800/2000, Train Loss: 0.7654
Epoch:1900/2000, Train Loss: 0.7612
Epoch:2000/2000, Train Loss: 0.7578


In [ ]:
model_4 = RNN()
model_4.to(device)

train(model_4, nn.CrossEntropyLoss(), optim.RAdam(model_4.parameters(), lr=0.01), X, 2000)

Epoch:100/2000, Train Loss: 0.9895
Epoch:200/2000, Train Loss: 0.9174
Epoch:300/2000, Train Loss: 0.6715
Epoch:400/2000, Train Loss: 0.6342
Epoch:500/2000, Train Loss: 0.6091
Epoch:600/2000, Train Loss: 0.5893
Epoch:700/2000, Train Loss: 0.5721
Epoch:800/2000, Train Loss: 0.5568
Epoch:900/2000, Train Loss: 0.5431
Epoch:1000/2000, Train Loss: 0.5301
Epoch:1100/2000, Train Loss: 0.5175
Epoch:1200/2000, Train Loss: 0.5059
Epoch:1300/2000, Train Loss: 0.4950
Epoch:1400/2000, Train Loss: 0.4819
Epoch:1500/2000, Train Loss: 0.4707
Epoch:1600/2000, Train Loss: 0.4596
Epoch:1700/2000, Train Loss: 0.4476
Epoch:1800/2000, Train Loss: 0.4366
Epoch:1900/2000, Train Loss: 0.4262
Epoch:2000/2000, Train Loss: 0.4156


In [ ]:
model_5 = RNN()
model_5.to(device)

train(model_5, nn.CrossEntropyLoss(), optim.Adam(model_5.parameters(), lr=0.1), X, 500)

Epoch:100/500, Train Loss: 1.1758
Epoch:200/500, Train Loss: 1.0180
Epoch:300/500, Train Loss: 2.2979
Epoch:400/500, Train Loss: 0.8912
Epoch:500/500, Train Loss: 1.0327


In [ ]:
def predict(model, characters):
    start = torch.zeros(size=(1, len(characters), vocab_size))
    for idx, ch in enumerate(characters):
        start[0][idx][ch_to_idx[ch]] = 1
    out, hidden = model(start)

    prob = nn.functional.softmax(out[-1], dim=0).data

    char_ind = torch.max(prob, dim=0)[1].item()

    return idx_to_ch[char_ind], hidden

def sample(model, start='h'):
    model.eval() # eval mode
    start = start.lower()
    chars = [ch for ch in start]
    size = 20 - len(chars)

    for ii in range(size):
        char, h = predict(model, chars)
        chars.append(char)

    return ''.join(chars).replace(' ', '')

In [2]:
sample(model, 'jr')

NameError: ignored

In [ ]:
sample(model_2, 'jr')

'jrmplat'

In [ ]:
sample(model_3, 'jr')

'jr'

In [ ]:
sample(model_4, 'jr')

'jranderforspiest'

In [ ]:
sample(model_5, 'jr')

'jra'